- Filename: ViT-GSOM_tuning_USPS.ipynb
- Author: Tomas Hanzlik <hanzlto3@fit.cvut.cz>
- Created: 2026-04-22
- Description: Hyperparameter tuning of the ViT-GSOM model on the USPS dataset

# ViT-GSOM hyperparameter tuning on USPS dataset

Hyperparameter tuning of the ViT-GSOM model to find the best setup using the optuna library

In [1]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import ConcatDataset

import numpy as np
import matplotlib.pyplot as plt

from ViTGSOM import AutoEncoder, ViTLossReconstruction, SomLoss
from help_functions import get_grid_coords, decay_exponential, calculate_QE_TE_Purity, plot_umap_som_weights, get_node_hits, plot_som_weights, plot_som_mnist, plot_som_pie_grid, generate_extended_u_matrix, visualize_u_matrix_extended 

import optuna
import copy

Download and concatenate the dataset

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(), 
])


train_set = datasets.USPS(root="./data", train=True, download=True, transform=transform)
test_set = datasets.USPS(root="./data", train=False, download=True, transform=transform)
dataset_complete = ConcatDataset([train_set, test_set])

Initialize the ViT-SOM

Details:
- Image size of 16
- Pixel size of 4
- 1 image channel
- Embedding dimension of 16
- Encoder depth of 4
- Decoder depth of 2
- 2 self-attention head
- MLP hidden dimension of size 64
- 20 Phase 1 epochs
- 500 Phase 2 epochs
- 100 Phase 3 epochs
- Learning rate of 0.0005
- Grow each grid after 10 epochs
- Initial 5 SOM rows
- Initial 5 SOM columns
- Stop Phase 2 at 0.90 Purity


In [3]:
config_usps = {
    'img_size': 16,
    'patch_size': 4,
    'num_of_channels': 1,
    'embed_dim': 16,
    'enc_depth': 4,
    'dec_depth': 2,
    'num_heads': 2,
    'mlp_dim': 64,
    'epochs_phase1': 20,
    'epochs_phase2': 500,
    'epochs_phase3': 100,
    'lr': 0.0005,
    'grow_after_epochs': 10,
    'som_rows': 5,
    'som_cols': 5,
    'stop_growth_purity': 0.90
}

In [4]:
def phase_1(model, device, loader, config):
    criterionViT = ViTLossReconstruction()
    
    # define parameters for which the gradients will be computed
    model.som_weights.requires_grad = False
    for param in model.encoder.parameters(): param.requires_grad = True
    for param in model.decoder.parameters(): param.requires_grad = True

    # define optimizer and scheduler with active parameters
    paramsViT = list(model.encoder.parameters()) + list(model.decoder.parameters())
    optimizerViT = optim.AdamW(paramsViT, lr=config['lr'])
    scheduler = CosineAnnealingLR(optimizerViT, T_max=config['epochs_phase1'])


    for epoch in range(config['epochs_phase1']):
        running_mse = 0.0

        for images, _ in loader:
            images = images.to(device)

            reconstructed, latent = model(images)
            som_weights = model.get_som_weights()

            # calculate loss
            l_nn = criterionViT(images, reconstructed)

            # update the active weights of the network
            optimizerViT.zero_grad()
            l_nn.backward()
            optimizerViT.step()
            running_mse += l_nn.item()

        scheduler.step()
        
def phase_2(model, device, loader, config):
    criterionSOM = SomLoss()
    
    # define parameters for which the gradients will be computed
    model.som_weights.requires_grad = True
    for param in model.encoder.parameters(): param.requires_grad = False
    for param in model.decoder.parameters(): param.requires_grad = False

    # define optimizer and scheduler with active parameters
    paramsSOM = model.som_weights
    optimizerSOM = optim.AdamW([paramsSOM], lr=config['lr'])
    scheduler = CosineAnnealingLR(optimizerSOM, T_max=config['epochs_phase2'])

    # starting and ending value of sigma, beta is calculated to reach sigma_end at last epoch
    sigma_start = model.get_sigma()
    sigma_end = 0.3
    beta = (sigma_end / sigma_start) ** (1 / config['grow_after_epochs'])
    epochs_since_reset = 0

    # get grid coordinates of each cell on the SOM grid
    rows, cols = model.get_som_shape() 
    grid_coords = get_grid_coords(rows, cols, device)

    unique_labels = set()

    for epoch in range(config['epochs_phase2']):

        running_som = 0.0

        sigma_t = decay_exponential(sigma_start, beta, epochs_since_reset)

        for images, labels in loader:
            images = images.to(device)
            unique_labels.update(labels.tolist())

            reconstructed, latent = model(images)
            som_weights = model.get_som_weights()

            # calculate loss
            l_som = criterionSOM(latent, som_weights, grid_coords, sigma_t)

            # update the active weights of the network
            optimizerSOM.zero_grad()
            l_som.backward()
            optimizerSOM.step()

            running_som += l_som.item()

        # updating learning rule through CosineAnnealingLR
        scheduler.step()

        metrics = calculate_QE_TE_Purity(model, loader, device, purity_only = True)

        if epoch > 0 and (epoch + 1) % config['grow_after_epochs'] == 0:

            if metrics["Purity"] > config['stop_growth_purity']:
                break

            epochs_since_reset += 1
            model.start_growth(loader, device)
            paramsSOM = model.get_som_weights()
            optimizerSOM = optim.AdamW([paramsSOM], lr=config['lr'])

            for param_group in optimizerSOM.param_groups: param_group['initial_lr'] = config['lr']

            scheduler = CosineAnnealingLR(optimizerSOM, T_max=config['epochs_phase2'], last_epoch=epoch)
            grid_coords = get_grid_coords(model.current_row_num, model.current_col_num, device)

            sigma_start = model.get_sigma()
            sigma_start = max(sigma_start, 2.0)
            beta = (sigma_end / sigma_start) ** (1 / max(1, config['grow_after_epochs']))
            epochs_since_reset = 0

        else:
            epochs_since_reset += 1
            
def phase_3(model, device, loader, config):
    criterionSOM = SomLoss()

    # define parameters for which the gradients will be computed
    model.som_weights.requires_grad = True
    for param in model.encoder.parameters():
        param.requires_grad = False
    for param in model.decoder.parameters():
        param.requires_grad = False

    # define optimizer and scheduler with active parameters
    paramsSOM = model.get_som_weights()
    optimizerSOM = optim.AdamW([paramsSOM], lr=config['lr'])
    scheduler = CosineAnnealingLR(optimizerSOM, T_max=config['epochs_phase3'], last_epoch=-1)

    # starting and ending value of sigma, beta is calculated to reach sigma_end at last epoch
    sigma_start = 2
    sigma_end = 0.01
    beta = (sigma_end / sigma_start) ** (1 / config['epochs_phase3'])

    # get grid coordinates of each cell on the SOM grid
    grid_coords = get_grid_coords(model.current_row_num, model.current_col_num, device)

    best_purity = 0.0

    for epoch in range(config['epochs_phase3']):

        running_som = 0.0

        sigma_t = decay_exponential(sigma_start, beta, epoch)

        for images, _ in loader:
            images = images.to(device)

            reconstructed, latent = model(images)
            som_weights = model.get_som_weights()

            # calculate loss
            l_som = criterionSOM(latent, som_weights, grid_coords, sigma_t)

            # update the active weights of the network
            optimizerSOM.zero_grad()
            l_som.backward()
            optimizerSOM.step()

            running_som += l_som.item()

        # updating learning rule through CosineAnnealingLR
        scheduler.step()

        metrics = calculate_QE_TE_Purity(model, loader, device, purity_only = True)

        if metrics["Purity"] > best_purity: best_purity = metrics["Purity"]

    return best_purity, model.current_col_num * model.current_row_num

def objective(trial, dataset, config):
    """
    Optuna objective
    :param trial: Number of trials
    :param dataset: Dataset
    :param config: Configuration of the model
    """

    #define dataloader and allow gpu computation
    loader = torch.utils.data.DataLoader(dataset=dataset, batch_size=32, shuffle=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    trial_config = copy.deepcopy(config)
    
    # define hyperparameters and their range of potential values
    trial_config['lr'] = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    trial_config['enc_depth'] = trial.suggest_int('enc_depth', 2, 10)
    trial_config['dec_depth'] = trial.suggest_int('dec_depth', 1, 6)
    trial_config['mlp_dim'] = trial.suggest_categorical('mlp_dim', [32, 64, 128, 256])
    trial_config['num_heads'] = trial.suggest_categorical('num_heads', [2, 4, 8])
    head_dim = trial.suggest_categorical('head_dim', [4, 8, 16]) 
    trial_config['embed_dim'] = trial_config['num_heads'] * head_dim
    
    autoencoder = AutoEncoder(img_size=trial_config['img_size'], 
                          patch_size=trial_config['patch_size'], 
                          num_of_channels=trial_config['num_of_channels'], 
                          embed_dim=trial_config['embed_dim'], 
                          enc_depth=trial_config['enc_depth'],                                      
                          dec_depth=trial_config['dec_depth'], 
                          num_heads=trial_config['num_heads'], 
                          mlp_dim=trial_config['mlp_dim'],
                          som_rows=trial_config['som_rows'],
                          som_cols=trial_config['som_cols'])
    
    autoencoder.to(device)
    autoencoder.train()
    
    try:
        phase_1(autoencoder, device, loader, trial_config)
        phase_2(autoencoder, device, loader, trial_config)
        final_purity, grid_size = phase_3(autoencoder, device, loader, trial_config)
    except Exception as e:
        print(e)
        raise optuna.exceptions.TrialPruned()
    
    # save these attributes
    trial.set_user_attr("final_purity", final_purity)
    trial.set_user_attr("final_grid", grid_size)

    # formula to calculate the final score
    score = final_purity - grid_size * 0.0005
    return score

In [5]:
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: objective(trial, dataset_complete, config_usps), n_trials=50)

print("\n==========================================")
print(f"Best Trial score: {study.best_trial.value}")
best_purity = study.best_trial.user_attrs.get("final_purity")
best_grid = study.best_trial.user_attrs.get("final_grid")
print(f"Purity Achieved: {best_purity:.6f}")
print(f"Grid Size: {best_grid} nodes")
print("Hyperparameters: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-01 21:36:36,736] A new study created in memory with name: no-name-f99ff292-38e0-48d1-874b-1dda4a57fc5e
[I 2026-04-01 23:16:42,751] Trial 0 finished with value: 0.4960340933534093 and parameters: {'lr': 8.918408122058891e-05, 'enc_depth': 10, 'dec_depth': 2, 'mlp_dim': 128, 'num_heads': 2, 'head_dim': 8}. Best is trial 0 with value: 0.4960340933534093.
[I 2026-04-02 00:15:28,596] Trial 1 finished with value: 0.1963755646375564 and parameters: {'lr': 1.1780736251225316e-05, 'enc_depth': 5, 'dec_depth': 5, 'mlp_dim': 64, 'num_heads': 8, 'head_dim': 16}. Best is trial 0 with value: 0.4960340933534093.
[I 2026-04-02 00:59:42,150] Trial 2 finished with value: 0.4364100881910088 and parameters: {'lr': 1.741645423553061e-05, 'enc_depth': 5, 'dec_depth': 1, 'mlp_dim': 128, 'num_heads': 4, 'head_dim': 8}. Best is trial 0 with value: 0.4960340933534093.
[I 2026-04-02 02:06:28,033] Trial 3 finished with value: 0.4856679931167993 and parameters: {'lr': 9.735275124044308e-05, 'enc_depth':


Best Trial score: 0.8469129920412992
Purity Achieved: 0.920413
Grid Size: 147 nodes
Hyperparameters: 
    lr: 0.0008064243890921968
    enc_depth: 4
    dec_depth: 5
    mlp_dim: 128
    num_heads: 4
    head_dim: 16
